In [1]:
import sys
sys.path.append('..')

import pandas as pd
from utils.db_utils import write_table, read_table

In [2]:
df = pd.read_csv("../../data/state_graduates/Johor.csv")
df.head(20)

,state,statistics,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Johor,total_graduate_degree,190.8,216.7,250.5,258.4,207.4,220.7,235.2,250.4,272.1
1,Johor,emp_graduate_degree,143.7,163.9,190.5,197.0,183.5,196.1,208.3,224.4,241.9
2,Johor,unemp_graduate_degree,8.2,7.2,8.2,7.9,10.4,10.8,10.0,9.1,10.4
3,Johor,outside_labour_force_degree,38.9,45.6,51.8,53.5,13.6,13.8,17.0,16.9,19.8
4,Johor,unemp_rate_degree,5.4,4.2,4.1,3.9,5.4,5.2,4.6,3.9,4.1
5,Johor,total_graduate_diploma,177.2,201.2,236.7,234.6,243.1,252.7,263.4,271.3,282.7
6,Johor,emp_graduate_diploma,152.5,171.8,203.9,203.6,202.5,213.6,221.7,230.1,238.8
7,Johor,unemp_graduate_diploma,7.5,8.3,5.4,6.7,7.6,6.6,7.4,6.1,5.9
8,Johor,outside_labour_force_diploma,17.2,21.1,27.4,24.3,33.0,32.4,34.3,35.2,38.1
9,Johor,unemp_rate_diploma,4.7,4.6,2.5,3.2,3.6,3.0,3.2,2.6,2.4


In [3]:
def transform_graduate_state_data(file_path):
    df = pd.read_csv(file_path)

    # reshape years into rows
    df_long = df.melt(
        id_vars=["state", "statistics"],
        var_name="year",
        value_name="value"
    )

    # extract metric and qualification from statistics
    def parse_stat(stat):
        if pd.isna(stat):
            return pd.Series([None, None], index=['metric', 'qualification'])
        parts = str(stat).rsplit('_', 1)
        if len(parts) == 2 and parts[1] in ['degree', 'diploma']:
            return pd.Series([parts[0], parts[1]], index=['metric', 'qualification'])
        return pd.Series([stat, None], index=['metric', 'qualification'])

    df_long[['metric', 'qualification']] = df_long['statistics'].apply(parse_stat)

    # pivot metrics into columns (one row per state/year/qualification)
    df_pivot = df_long.pivot_table(
        index=['state', 'year', 'qualification'],
        columns='metric',
        values='value',
        aggfunc='first'
    ).reset_index()

    df_pivot.columns.name = None

    num_cols = [c for c in df_pivot.columns if c not in ['state', 'year', 'qualification']]
    for col in num_cols:
        df_pivot[col] = pd.to_numeric(df_pivot[col], errors='coerce')
        if col.endswith('_rate'):
            continue
        df_pivot[col] = df_pivot[col] * 1000

    df_pivot = df_pivot.sort_values(['state', 'year', 'qualification']).reset_index(drop=True)
    return df_pivot

In [4]:
df_johor = transform_graduate_state_data("../../data/state_graduates/Johor.csv")
df_johor.head(10)

,state,year,qualification,emp_graduate,outside_labour_force,total_graduate,unemp_graduate,unemp_rate
0,Johor,2016,degree,143700.0,38900.0,190800.0,8200.0,5.4
1,Johor,2016,diploma,152500.0,17200.0,177200.0,7500.0,4.7
2,Johor,2017,degree,163900.0,45600.0,216700.0,7200.0,4.2
3,Johor,2017,diploma,171800.0,21100.0,201200.0,8300.0,4.6
4,Johor,2018,degree,190500.0,51800.0,250500.0,8200.0,4.1
5,Johor,2018,diploma,203900.0,27400.0,236700.0,5400.0,2.5
6,Johor,2019,degree,197000.0,53500.0,258400.0,7900.0,3.9
7,Johor,2019,diploma,203600.0,24300.0,234600.0,6700.0,3.2
8,Johor,2020,degree,183500.0,13600.0,207400.0,10400.0,5.4
9,Johor,2020,diploma,202500.0,33000.0,243100.0,7600.0,3.6


In [5]:
df_kedah = transform_graduate_state_data("../../data/state_graduates/Kedah.csv")
df_kedah.head(10)

,state,year,qualification,emp_graduate,outside_labour_force,total_graduate,unemp_graduate,unemp_rate
0,Kedah,2016,degree,86300.0,26200.0,116700.0,4200.0,4.6
1,Kedah,2016,diploma,77600.0,16100.0,95800.0,2100.0,2.6
2,Kedah,2017,degree,94400.0,25800.0,124400.0,4200.0,4.3
3,Kedah,2017,diploma,75800.0,16300.0,95500.0,3400.0,4.2
4,Kedah,2018,degree,107800.0,32000.0,144900.0,5100.0,4.5
5,Kedah,2018,diploma,100500.0,19000.0,121800.0,2300.0,2.1
6,Kedah,2019,degree,108600.0,35000.0,149100.0,5500.0,4.8
7,Kedah,2019,diploma,104900.0,18100.0,126800.0,3800.0,3.5
8,Kedah,2020,degree,107700.0,12800.0,126100.0,5500.0,4.9
9,Kedah,2020,diploma,93900.0,31900.0,129000.0,3200.0,3.3


In [6]:
df_kelantan = transform_graduate_state_data("../../data/state_graduates/Kelantan.csv")
df_kelantan.head(10)

,state,year,qualification,emp_graduate,outside_labour_force,total_graduate,unemp_graduate,unemp_rate
0,Kelantan,2016,degree,72200.0,22300.0,99200.0,4700.0,6.1
1,Kelantan,2016,diploma,58900.0,14100.0,76500.0,3500.0,5.6
2,Kelantan,2017,degree,70900.0,26200.0,102700.0,5600.0,7.3
3,Kelantan,2017,diploma,60900.0,15600.0,81300.0,4800.0,7.3
4,Kelantan,2018,degree,85500.0,29000.0,121300.0,6700.0,7.3
5,Kelantan,2018,diploma,71800.0,17600.0,92700.0,3300.0,4.4
6,Kelantan,2019,degree,90400.0,32700.0,130800.0,7700.0,7.8
7,Kelantan,2019,diploma,78500.0,15900.0,99400.0,5000.0,6.0
8,Kelantan,2020,degree,75600.0,12300.0,92700.0,4800.0,5.9
9,Kelantan,2020,diploma,60000.0,25100.0,89000.0,4000.0,6.3


In [7]:
df_kl = transform_graduate_state_data("../../data/state_graduates/Kuala Lumpur.csv")
df_kl.head(10)

,state,year,qualification,emp_graduate,outside_labour_force,total_graduate,unemp_graduate,unemp_rate
0,W.P. Kuala Lumpur,2016,degree,187500.0,34700.0,228100.0,5900.0,3.0
1,W.P. Kuala Lumpur,2016,diploma,162700.0,54700.0,224600.0,7200.0,4.2
2,W.P. Kuala Lumpur,2017,degree,180700.0,30600.0,216100.0,4800.0,2.6
3,W.P. Kuala Lumpur,2017,diploma,148800.0,47400.0,201000.0,4800.0,3.1
4,W.P. Kuala Lumpur,2018,degree,209800.0,36700.0,248800.0,2300.0,1.1
5,W.P. Kuala Lumpur,2018,diploma,182900.0,66800.0,255600.0,5900.0,3.1
6,W.P. Kuala Lumpur,2019,degree,194900.0,24900.0,223400.0,3600.0,1.9
7,W.P. Kuala Lumpur,2019,diploma,172200.0,46300.0,223300.0,4800.0,2.7
8,W.P. Kuala Lumpur,2020,degree,284800.0,39200.0,331300.0,7300.0,2.5
9,W.P. Kuala Lumpur,2020,diploma,164700.0,35400.0,206800.0,6800.0,3.9


In [8]:
df_labuan = transform_graduate_state_data("../../data/state_graduates/Labuan.csv")
df_labuan.head(10)

,state,year,qualification,emp_graduate,outside_labour_force,total_graduate,unemp_graduate,unemp_rate
0,W.P. Labuan,2016,degree,5900.0,400.0,6700.0,400.0,5.9
1,W.P. Labuan,2016,diploma,4400.0,2000.0,6700.0,400.0,7.4
2,W.P. Labuan,2017,degree,4700.0,300.0,5200.0,200.0,4.1
3,W.P. Labuan,2017,diploma,4000.0,1600.0,6000.0,400.0,9.1
4,W.P. Labuan,2018,degree,5300.0,400.0,6000.0,300.0,5.4
5,W.P. Labuan,2018,diploma,5100.0,1800.0,7200.0,300.0,5.6
6,W.P. Labuan,2019,degree,6200.0,500.0,7100.0,400.0,6.1
7,W.P. Labuan,2019,diploma,5500.0,1700.0,7300.0,100.0,1.8
8,W.P. Labuan,2020,degree,3600.0,400.0,4100.0,100.0,2.0
9,W.P. Labuan,2020,diploma,5500.0,1200.0,6900.0,100.0,2.5


In [9]:
df_melaka = transform_graduate_state_data("../../data/state_graduates/Melaka.csv")
df_melaka.head(10)

,state,year,qualification,emp_graduate,outside_labour_force,total_graduate,unemp_graduate,unemp_rate
0,Melaka,2016,degree,51300.0,16800.0,69000.0,900.0,1.7
1,Melaka,2016,diploma,49900.0,8200.0,58500.0,400.0,0.8
2,Melaka,2017,degree,60900.0,18100.0,80000.0,1100.0,1.8
3,Melaka,2017,diploma,59100.0,12700.0,72500.0,700.0,1.2
4,Melaka,2018,degree,68500.0,19000.0,89100.0,1600.0,2.3
5,Melaka,2018,diploma,61700.0,13400.0,75400.0,300.0,0.5
6,Melaka,2019,degree,75300.0,21200.0,98400.0,1900.0,2.5
7,Melaka,2019,diploma,68600.0,12500.0,81700.0,600.0,0.9
8,Melaka,2020,degree,65800.0,9300.0,77000.0,1900.0,2.8
9,Melaka,2020,diploma,72400.0,19000.0,92600.0,1200.0,1.6


In [10]:
df_ns = transform_graduate_state_data("../../data/state_graduates/Negeri Sembilan.csv")
df_ns.head(10)

,state,year,qualification,emp_graduate,outside_labour_force,total_graduate,unemp_graduate,unemp_rate
0,Negeri Sembilan,2016,degree,56600.0,7700.0,66800.0,2400.0,4.1
1,Negeri Sembilan,2016,diploma,56000.0,14300.0,73300.0,3000.0,5.0
2,Negeri Sembilan,2017,degree,68800.0,11300.0,82000.0,1900.0,2.7
3,Negeri Sembilan,2017,diploma,60500.0,19100.0,82500.0,2900.0,4.6
4,Negeri Sembilan,2018,degree,67900.0,11300.0,82300.0,3100.0,4.4
5,Negeri Sembilan,2018,diploma,69000.0,20000.0,91500.0,2500.0,3.5
6,Negeri Sembilan,2019,degree,73500.0,12000.0,88400.0,2900.0,3.8
7,Negeri Sembilan,2019,diploma,72200.0,20400.0,95700.0,3200.0,4.2
8,Negeri Sembilan,2020,degree,79600.0,10600.0,93900.0,3700.0,4.4
9,Negeri Sembilan,2020,diploma,75800.0,25700.0,104400.0,2900.0,3.7


In [11]:
df_pahang = transform_graduate_state_data("../../data/state_graduates/Pahang.csv")
df_pahang.head(10)

,state,year,qualification,emp_graduate,outside_labour_force,total_graduate,unemp_graduate,unemp_rate
0,Pahang,2016,degree,68000.0,8100.0,78300.0,2200.0,3.1
1,Pahang,2016,diploma,72300.0,15400.0,91900.0,4200.0,5.5
2,Pahang,2017,degree,63400.0,9900.0,76200.0,2900.0,4.4
3,Pahang,2017,diploma,70800.0,17700.0,92700.0,4200.0,5.6
4,Pahang,2018,degree,71800.0,10700.0,84700.0,2200.0,3.0
5,Pahang,2018,diploma,79400.0,20800.0,103800.0,3600.0,4.3
6,Pahang,2019,degree,81300.0,11900.0,96400.0,3200.0,3.8
7,Pahang,2019,diploma,88600.0,28900.0,122600.0,5100.0,5.4
8,Pahang,2020,degree,78200.0,8300.0,89400.0,2900.0,3.6
9,Pahang,2020,diploma,73500.0,20400.0,97500.0,3600.0,4.6


In [12]:
df_penang = transform_graduate_state_data("../../data/state_graduates/Pulau Pinang.csv")
df_penang.head(10)

,state,year,qualification,emp_graduate,outside_labour_force,total_graduate,unemp_graduate,unemp_rate
0,Pulau Pinang,2016,degree,126700.0,16500.0,146900.0,3700.0,2.8
1,Pulau Pinang,2016,diploma,119600.0,29000.0,151500.0,2900.0,2.4
2,Pulau Pinang,2017,degree,127900.0,20600.0,150600.0,2200.0,1.7
3,Pulau Pinang,2017,diploma,116500.0,34800.0,155600.0,4300.0,3.6
4,Pulau Pinang,2018,degree,132100.0,21800.0,157500.0,3600.0,2.7
5,Pulau Pinang,2018,diploma,133000.0,37200.0,173600.0,3400.0,2.5
6,Pulau Pinang,2019,degree,147900.0,23600.0,173700.0,2200.0,1.5
7,Pulau Pinang,2019,diploma,133400.0,44500.0,181100.0,3200.0,2.3
8,Pulau Pinang,2020,degree,166400.0,21000.0,191000.0,3500.0,2.1
9,Pulau Pinang,2020,diploma,105100.0,35900.0,144300.0,3300.0,3.0


In [13]:
df_perak = transform_graduate_state_data("../../data/state_graduates/Perak.csv")
df_perak.head(10)

,state,year,qualification,emp_graduate,outside_labour_force,total_graduate,unemp_graduate,unemp_rate
0,Perak,2016,degree,104100.0,26000.0,135700.0,5700.0,5.2
1,Perak,2016,diploma,95800.0,19500.0,118000.0,2600.0,2.7
2,Perak,2017,degree,106000.0,36900.0,150500.0,7600.0,6.7
3,Perak,2017,diploma,102300.0,22100.0,131100.0,6700.0,6.1
4,Perak,2018,degree,115300.0,36000.0,157400.0,6100.0,5.0
5,Perak,2018,diploma,106200.0,24900.0,137500.0,6500.0,5.8
6,Perak,2019,degree,123100.0,45600.0,176900.0,8200.0,6.2
7,Perak,2019,diploma,121800.0,26600.0,153500.0,5000.0,3.9
8,Perak,2020,degree,110900.0,25300.0,144600.0,8500.0,7.1
9,Perak,2020,diploma,111100.0,40400.0,156500.0,5100.0,4.4


In [14]:
df_perlis = transform_graduate_state_data("../../data/state_graduates/Perlis.csv")
df_perlis.head(10)

,state,year,qualification,emp_graduate,outside_labour_force,total_graduate,unemp_graduate,unemp_rate
0,Perlis,2016,degree,10600.0,3200.0,14100.0,300.0,2.7
1,Perlis,2016,diploma,11800.0,4500.0,16700.0,400.0,3.5
2,Perlis,2017,degree,12200.0,4500.0,17300.0,600.0,4.7
3,Perlis,2017,diploma,12200.0,5600.0,18700.0,900.0,6.9
4,Perlis,2018,degree,11600.0,2400.0,14500.0,500.0,4.1
5,Perlis,2018,diploma,12600.0,5900.0,19700.0,1200.0,8.7
6,Perlis,2019,degree,12900.0,2900.0,16400.0,600.0,4.4
7,Perlis,2019,diploma,15800.0,6700.0,23800.0,1300.0,7.6
8,Perlis,2020,degree,15100.0,3800.0,19800.0,900.0,5.8
9,Perlis,2020,diploma,13400.0,6400.0,20500.0,800.0,5.5


In [15]:
df_putrajaya = transform_graduate_state_data("../../data/state_graduates/Putrajaya.csv")
df_putrajaya.head(10)

,state,year,qualification,emp_graduate,outside_labour_force,total_graduate,unemp_graduate,unemp_rate
0,W.P. Putrajaya,2016,degree,14500.0,1900.0,16700.0,300.0,1.7
1,W.P. Putrajaya,2016,diploma,10400.0,700.0,11300.0,200.0,1.9
2,W.P. Putrajaya,2017,degree,14500.0,1900.0,16700.0,300.0,2.0
3,W.P. Putrajaya,2017,diploma,10000.0,1100.0,11200.0,100.0,1.0
4,W.P. Putrajaya,2018,degree,15500.0,1700.0,17300.0,100.0,0.6
5,W.P. Putrajaya,2018,diploma,9900.0,1500.0,11500.0,200.0,2.0
6,W.P. Putrajaya,2019,degree,14700.0,2700.0,17700.0,200.0,1.3
7,W.P. Putrajaya,2019,diploma,10500.0,1400.0,12000.0,100.0,0.9
8,W.P. Putrajaya,2020,degree,15100.0,2800.0,18100.0,200.0,1.3
9,W.P. Putrajaya,2020,diploma,14600.0,2200.0,17100.0,200.0,1.6


In [16]:
df_sabah = transform_graduate_state_data("../../data/state_graduates/Sabah.csv")
df_sabah.head(10)

,state,year,qualification,emp_graduate,outside_labour_force,total_graduate,unemp_graduate,unemp_rate
0,Sabah,2016,degree,110300.0,23600.0,144100.0,10100.0,8.4
1,Sabah,2016,diploma,107800.0,11100.0,125200.0,6300.0,5.5
2,Sabah,2017,degree,121400.0,32100.0,163500.0,10000.0,7.6
3,Sabah,2017,diploma,115700.0,14800.0,141900.0,11400.0,9.0
4,Sabah,2018,degree,120400.0,32100.0,164400.0,12000.0,9.1
5,Sabah,2018,diploma,120600.0,12000.0,143300.0,10700.0,8.1
6,Sabah,2019,degree,139000.0,36100.0,188000.0,12900.0,8.5
7,Sabah,2019,diploma,130100.0,9700.0,150500.0,10700.0,7.6
8,Sabah,2020,degree,128400.0,11200.0,152600.0,12900.0,9.1
9,Sabah,2020,diploma,113500.0,25600.0,151300.0,12200.0,9.7


In [17]:
df_sarawak = transform_graduate_state_data("../../data/state_graduates/Sarawak.csv")
df_sarawak.head(10)

,state,year,qualification,emp_graduate,outside_labour_force,total_graduate,unemp_graduate,unemp_rate
0,Sarawak,2016,degree,98000.0,12600.0,115800.0,5200.0,5.0
1,Sarawak,2016,diploma,101700.0,19800.0,126500.0,5000.0,4.6
2,Sarawak,2017,degree,99300.0,13300.0,117900.0,5200.0,5.0
3,Sarawak,2017,diploma,105000.0,22400.0,133100.0,5700.0,5.1
4,Sarawak,2018,degree,113400.0,17500.0,136600.0,5700.0,4.8
5,Sarawak,2018,diploma,116400.0,26400.0,148700.0,6000.0,4.9
6,Sarawak,2019,degree,125300.0,16800.0,147600.0,5500.0,4.2
7,Sarawak,2019,diploma,126700.0,29200.0,164400.0,8500.0,6.3
8,Sarawak,2020,degree,108000.0,11300.0,123600.0,4300.0,3.8
9,Sarawak,2020,diploma,94500.0,26700.0,126300.0,5100.0,5.1


In [18]:
df_selangor = transform_graduate_state_data("../../data/state_graduates/Selangor.csv")
df_selangor.head(10)

,state,year,qualification,emp_graduate,outside_labour_force,total_graduate,unemp_graduate,unemp_rate
0,Selangor,2016,degree,573000.0,96900.0,690200.0,20300.0,3.4
1,Selangor,2016,diploma,582400.0,70000.0,672100.0,19700.0,3.3
2,Selangor,2017,degree,638300.0,118100.0,775200.0,18800.0,2.9
3,Selangor,2017,diploma,628100.0,88900.0,733400.0,16500.0,2.6
4,Selangor,2018,degree,639400.0,98000.0,760900.0,23500.0,3.5
5,Selangor,2018,diploma,623100.0,73800.0,716200.0,19300.0,3.0
6,Selangor,2019,degree,700700.0,128700.0,849900.0,20500.0,2.8
7,Selangor,2019,diploma,684400.0,94700.0,801300.0,22200.0,3.1
8,Selangor,2020,degree,776500.0,93100.0,897000.0,27400.0,5.6
9,Selangor,2020,diploma,548100.0,111100.0,691600.0,32400.0,5.6


In [19]:
df_terengganu = transform_graduate_state_data("../../data/state_graduates/Terengganu.csv")
df_terengganu.head(10)

,state,year,qualification,emp_graduate,outside_labour_force,total_graduate,unemp_graduate,unemp_rate
0,Terengganu,2016,degree,50400.0,7300.0,60700.0,2900.0,5.5
1,Terengganu,2016,diploma,53600.0,19700.0,76600.0,3200.0,5.7
2,Terengganu,2017,degree,52800.0,9100.0,64700.0,2800.0,5.0
3,Terengganu,2017,diploma,58700.0,21500.0,84300.0,4100.0,6.5
4,Terengganu,2018,degree,60300.0,10700.0,75100.0,4100.0,6.4
5,Terengganu,2018,diploma,63100.0,29200.0,97800.0,5500.0,8.0
6,Terengganu,2019,degree,69700.0,11900.0,84600.0,3100.0,4.3
7,Terengganu,2019,diploma,69400.0,29200.0,102200.0,3600.0,4.9
8,Terengganu,2020,degree,51300.0,10400.0,64300.0,2600.0,4.7
9,Terengganu,2020,diploma,54000.0,20200.0,77400.0,3200.0,5.7


In [20]:
df_merged = pd.concat([
    df_johor, df_kedah, df_kelantan, df_kl, df_labuan, df_melaka, df_ns, df_pahang, df_penang, df_perak, df_perlis,
    df_putrajaya, df_sabah, df_sarawak, df_selangor, df_terengganu
], ignore_index=True)
write_table(df_merged, "sc_bronze", "dosm_graduates_state")
df_merged.head(10)

Table sc_bronze.dosm_graduates_state written successfully.


,state,year,qualification,emp_graduate,outside_labour_force,total_graduate,unemp_graduate,unemp_rate
0,Johor,2016,degree,143700.0,38900.0,190800.0,8200.0,5.4
1,Johor,2016,diploma,152500.0,17200.0,177200.0,7500.0,4.7
2,Johor,2017,degree,163900.0,45600.0,216700.0,7200.0,4.2
3,Johor,2017,diploma,171800.0,21100.0,201200.0,8300.0,4.6
4,Johor,2018,degree,190500.0,51800.0,250500.0,8200.0,4.1
5,Johor,2018,diploma,203900.0,27400.0,236700.0,5400.0,2.5
6,Johor,2019,degree,197000.0,53500.0,258400.0,7900.0,3.9
7,Johor,2019,diploma,203600.0,24300.0,234600.0,6700.0,3.2
8,Johor,2020,degree,183500.0,13600.0,207400.0,10400.0,5.4
9,Johor,2020,diploma,202500.0,33000.0,243100.0,7600.0,3.6
